In [ ]:
import pandas as pd
import os
from openai import OpenAI
import json
import re
import time

In [ ]:
!pip install -q rouge-score
# Lightweight ROUGE without evaluate/datasets (avoids pyarrow issues)
from rouge_score import rouge_scorer
import numpy as np

In [2]:
# Kaggle datasets are mounted under /kaggle/input/<dataset-name>/
DATA_DIR = "/kaggle/input/slm-review-comments"

files = {
    "gemma_few": "gemma_few_shot_results.csv",
    "gemma_zero": "gemma_zero_shot_results.csv",
    "gpt_few": "gpt_few_shot_results.csv",
    "gpt_zero": "gpt_zero_shot_results.csv",
    "mistral_few": "mistral_few_shot_results.csv",
    "mistral_zero": "mistral_zero_shot_results.csv",
    "phi_few": "phi_few_shot_results.csv",
    "phi_zero": "phi_zero_shot_results.csv",
}

dfs = {}
for key, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    dfs[key] = pd.read_csv(path)
    print(f"{key}: loaded {fname} | shape={dfs[key].shape}")

# Optional quick peek to confirm columns
for key, df in dfs.items():
    print(f"\n=== {key} columns ===")
    print(df.columns.tolist())
    print(df.head(1))


gemma_few: loaded gemma_few_shot_results.csv | shape=(100, 11)
gemma_zero: loaded gemma_zero_shot_results.csv | shape=(100, 11)
gpt_few: loaded gpt_few_shot_results.csv | shape=(100, 11)
gpt_zero: loaded gpt_zero_shot_results.csv | shape=(100, 11)
mistral_few: loaded mistral_few_shot_results.csv | shape=(100, 11)
mistral_zero: loaded mistral_zero_shot_results.csv | shape=(100, 11)
phi_few: loaded phi_few_shot_results.csv | shape=(100, 11)
phi_zero: loaded phi_zero_shot_results.csv | shape=(100, 11)

=== gemma_few columns ===
['comment', 'comment_created', 'model_name', 'prompt_type', 'elapsed_s', 'input_tokens', 'output_tokens', 'total_tokens', 'avg_power_w', 'energy_joules', 'energy_wh']
                                             comment  \
0  Is the path user provided or  autogenerated ? ...   

                                     comment_created model_name prompt_type  \
0  [\n  {\n    "line": 203,\n    "type": "logic",...      gemma    few_shot   

   elapsed_s  input_tokens  ou

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

In [3]:
api_key = "OPENAI_API_KEY"


API_KEY = api_key 
client = OpenAI(api_key=API_KEY)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ],
    max_tokens=20,
)

print(response.choices[0].message.content.strip())


2 + 2 equals 4.


In [11]:
def extract_comment_text(x):
    """
    Normalize extracted review text:
    - Handles JSON arrays with multiple items
    - Removes leading `json`, ```json fences, etc.
    - Pulls "comment" fields out safely
    """
    if x is None:
        return ""

    if not isinstance(x, str):
        x = str(x)

    s = x.strip()
    if s == "":
        return ""

    # Remove accidental "```json", "json", "```" prefixes
    s = s.lstrip("`").strip()
    if s.lower().startswith("json"):
        s = s[4:].strip()

    # Try to find JSON inside (regex: match the first [ ... ] block)
    import re
    json_match = re.search(r'(\[.*\])', s, flags=re.DOTALL)
    if json_match:
        s = json_match.group(1).strip()

    # Try JSON parsing
    try:
        obj = json.loads(s)

        if isinstance(obj, list):
            comments = []
            for item in obj:
                if isinstance(item, dict) and "comment" in item:
                    comments.append(str(item["comment"]).strip())
            if comments:
                return " ; ".join(comments)

        if isinstance(obj, dict) and "comment" in obj:
            return str(obj["comment"]).strip()

    except Exception:
        # If parsing fails, fallback to raw text
        pass

    # Final fallback: return cleaned string
    return s
 


In [20]:
JUDGE_SYSTEM_PROMPT_BINARY_SOFT = """
You are aM evaluator of code review comments.

You will be given:
1) a ground-truth review comment (human)
2) a model-generated review comment (may contain multiple points)

Your task:
- Output 1 if the model comment captures general idea of the same issue, context, risk, or suggestion as the ground truth.
- Output 0 if the model comment is unrelated to the ground truth.

Return ONLY one number: 0 or 1. No explanation, no extra text.
""".strip()


def _parse_binary_score(text):
    if text is None:
        return None
    t = text.strip()
    if t in {"0", "1"}:
        return int(t)
    m = re.search(r'\b(0|1)\b', t)
    return int(m.group(1)) if m else None


def gpt_judge_score_binary(gt_comment, pred_comment, model="gpt-4o", max_retries=3):

    gt_text = extract_comment_text(gt_comment)
    pred_text = extract_comment_text(pred_comment)

    if gt_text == "" or pred_text == "":
        return 0

    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT_BINARY_SOFT},
        {"role": "user", "content": f"Ground truth comment:\n{gt_text}\n\nModel comment:\n{pred_text}\n\nScore (0 or 1):"}
    ]

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0.0,
                max_tokens=3,
            )
            raw = resp.choices[0].message.content
            score = _parse_binary_score(raw)
            if score in {0, 1}:
                return score
            last_err = ValueError(f"Unparseable binary score: {raw}")

        except Exception as e:
            last_err = e

        time.sleep(1.2 * attempt)

    print("Binary judge failed:", repr(last_err))
    return None

In [22]:
judged_dfs = {}

for name, df in dfs.items():
    print(f"\n=== Processing {name} ({len(df)} rows) ===")
    df_judged = df.copy()

    scores = []
    for i, row in df_judged.iterrows():
        gt = row.get("comment", "")
        pred = row.get("comment_created", "")

        score = gpt_judge_score_binary(gt, pred)
        scores.append(score)

        print(f"  Row {i}/{len(df)} -> score={score}")

        time.sleep(0.2)  # rate limit safety

    df_judged["judge_score"] = scores
    judged_dfs[name] = df_judged
    print(f"✓ Finished {name}. Added binary judge_score.")



=== Processing gemma_few (100 rows) ===
  Row 0/100 -> score=0
  Row 1/100 -> score=1
  Row 2/100 -> score=1
  Row 3/100 -> score=0
  Row 4/100 -> score=0
  Row 5/100 -> score=0
  Row 6/100 -> score=0
  Row 7/100 -> score=1
  Row 8/100 -> score=0
  Row 9/100 -> score=1
  Row 10/100 -> score=0
  Row 11/100 -> score=0
  Row 12/100 -> score=0
  Row 13/100 -> score=0
  Row 14/100 -> score=0
  Row 15/100 -> score=1
  Row 16/100 -> score=1
  Row 17/100 -> score=1
  Row 18/100 -> score=0
  Row 19/100 -> score=1
  Row 20/100 -> score=0
  Row 21/100 -> score=0
  Row 22/100 -> score=1
  Row 23/100 -> score=0
  Row 24/100 -> score=0
  Row 25/100 -> score=0
  Row 26/100 -> score=0
  Row 27/100 -> score=0
  Row 28/100 -> score=0
  Row 29/100 -> score=0
  Row 30/100 -> score=1
  Row 31/100 -> score=0
  Row 32/100 -> score=0
  Row 33/100 -> score=1
  Row 34/100 -> score=0
  Row 35/100 -> score=1
  Row 36/100 -> score=1
  Row 37/100 -> score=0
  Row 38/100 -> score=0
  Row 39/100 -> score=1
  Row 40/

In [35]:
# Assumes you already have extract_comment_text(x) defined

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge_rowwise(df):
    r1_list, r2_list, rL_list = [], [], []

    for _, row in df.iterrows():
        gt = extract_comment_text(row.get("comment", ""))
        pred = extract_comment_text(row.get("comment_created", ""))

        if gt.strip() == "" or pred.strip() == "":
            r1_list.append(0.0)
            r2_list.append(0.0)
            rL_list.append(0.0)
            continue

        try:
            scores = scorer.score(gt, pred)  # returns precision/recall/fmeasure
            r1_list.append(float(scores["rouge1"].fmeasure))
            r2_list.append(float(scores["rouge2"].fmeasure))
            rL_list.append(float(scores["rougeL"].fmeasure))
        except Exception:
            r1_list.append(np.nan)
            r2_list.append(np.nan)
            rL_list.append(np.nan)

    out = df.copy()
    out["rouge1"] = r1_list
    out["rouge2"] = r2_list
    out["rougeL"] = rL_list
    return out


# Run for all 8 dataframes
rouged_dfs = {}
for name, df in dfs.items():
    print(f"Computing ROUGE for {name} ...")
    rouged_dfs[name] = compute_rouge_rowwise(df)
    print(f"Done {name}. Added: rouge1, rouge2, rougeL")

# Optional peek
for name, df in rouged_dfs.items():
    print(name, df[["rouge1", "rouge2", "rougeL"]].head(3))


Computing ROUGE for gemma_few ...
Done gemma_few. Added: rouge1, rouge2, rougeL
Computing ROUGE for gemma_zero ...
Done gemma_zero. Added: rouge1, rouge2, rougeL
Computing ROUGE for gpt_few ...
Done gpt_few. Added: rouge1, rouge2, rougeL
Computing ROUGE for gpt_zero ...
Done gpt_zero. Added: rouge1, rouge2, rougeL
Computing ROUGE for mistral_few ...
Done mistral_few. Added: rouge1, rouge2, rougeL
Computing ROUGE for mistral_zero ...
Done mistral_zero. Added: rouge1, rouge2, rougeL
Computing ROUGE for phi_few ...
Done phi_few. Added: rouge1, rouge2, rougeL
Computing ROUGE for phi_zero ...
Done phi_zero. Added: rouge1, rouge2, rougeL
gemma_few      rouge1    rouge2    rougeL
0  0.111111  0.000000  0.111111
1  0.211538  0.058824  0.134615
2  0.133333  0.000000  0.088889
gemma_zero      rouge1    rouge2    rougeL
0  0.163265  0.000000  0.122449
1  0.255639  0.030534  0.180451
2  0.111111  0.000000  0.074074
gpt_few      rouge1    rouge2    rougeL
0  0.057143  0.000000  0.057143
1  0.230088

In [36]:
rouged_dfs

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

{'gemma_few':                                               comment  \
 0   Is the path user provided or  autogenerated ? ...   
 1   The "False and" looks like a temporary stopgap...   
 2   This is really long, and will affect server st...   
 3   I wonder if the set comprehension here could h...   
 4                                      Indent problem   
 ..                                                ...   
 95  what is this line doing? Is it overwriting the...   
 96                   please add this to the docstring   
 97  Using a `set()` might change the order of the ...   
 98  Looks good! Please add an inspected key for ve...   
 99                      Why not use self.logger.info?   
 
                                       comment_created model_name prompt_type  \
 0   [\n  {\n    "line": 203,\n    "type": "logic",...      gemma    few_shot   
 1   [\n  {\n    "line": 1914,\n    "type": "logic"...      gemma    few_shot   
 2   ```json\n[\n  {\n    "line": 245,\n    "t

In [37]:
combined_dfs = {}

for name in dfs.keys():
    df_r = rouged_dfs[name]      # contains rouge1, rouge2, rougeL
    df_j = judged_dfs[name]      # contains judge_score

    # Merge by index since rows match 1-to-1
    merged = df_r.copy()
    merged["judge_score"] = df_j["judge_score"].values

    combined_dfs[name] = merged
    print(f"✓ Combined: {name} → columns now: {merged.columns.tolist()}")


✓ Combined: gemma_few → columns now: ['comment', 'comment_created', 'model_name', 'prompt_type', 'elapsed_s', 'input_tokens', 'output_tokens', 'total_tokens', 'avg_power_w', 'energy_joules', 'energy_wh', 'rouge1', 'rouge2', 'rougeL', 'judge_score']
✓ Combined: gemma_zero → columns now: ['comment', 'comment_created', 'model_name', 'prompt_type', 'elapsed_s', 'input_tokens', 'output_tokens', 'total_tokens', 'avg_power_w', 'energy_joules', 'energy_wh', 'rouge1', 'rouge2', 'rougeL', 'judge_score']
✓ Combined: gpt_few → columns now: ['comment', 'comment_created', 'model_name', 'prompt_type', 'elapsed_s', 'input_tokens', 'output_tokens', 'total_tokens', 'avg_power_w', 'energy_joules', 'energy_wh', 'rouge1', 'rouge2', 'rougeL', 'judge_score']
✓ Combined: gpt_zero → columns now: ['comment', 'comment_created', 'model_name', 'prompt_type', 'elapsed_s', 'input_tokens', 'output_tokens', 'total_tokens', 'avg_power_w', 'energy_joules', 'energy_wh', 'rouge1', 'rouge2', 'rougeL', 'judge_score']
✓ Comb

In [42]:
combined_dfs["phi_few"].describe()


,elapsed_s,input_tokens,output_tokens,total_tokens,avg_power_w,energy_joules,energy_wh,rouge1,rouge2,rougeL,judge_score
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,14.804070,1388.520000,147.510000,1536.030000,55.838165,832.989043,0.231386,0.110117,0.020902,0.087943,0.360000
std,7.733972,549.140384,90.408779,572.012246,2.134095,441.780007,0.122717,0.097161,0.049093,0.080090,0.482418
min,6.206866,925.000000,45.000000,989.000000,49.525510,333.765357,0.092713,0.000000,0.000000,0.000000,0.000000
25%,8.055479,1042.250000,56.000000,1158.750000,54.507642,436.463206,0.121240,0.036325,0.000000,0.031593,0.000000
50%,13.361880,1177.000000,112.000000,1345.500000,55.799506,740.927050,0.205813,0.094359,0.000000,0.071429,0.000000
75%,19.316858,1494.000000,256.000000,1733.250000,57.702155,1130.381878,0.313995,0.170517,0.020060,0.125665,1.000000
max,45.862779,3900.000000,256.000000,4156.000000,59.621827,2513.979489,0.698328,0.562500,0.333333,0.437500,1.000000


In [45]:
# Directory to save output
OUTDIR = "/kaggle/working/final_results"
os.makedirs(OUTDIR, exist_ok=True)

for name, df in combined_dfs.items():
    out_path = os.path.join(OUTDIR, f"{name}.csv")
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")


Saved: /kaggle/working/final_swapped_results/gemma_few.csv
Saved: /kaggle/working/final_swapped_results/gemma_zero.csv
Saved: /kaggle/working/final_swapped_results/gpt_few.csv
Saved: /kaggle/working/final_swapped_results/gpt_zero.csv
Saved: /kaggle/working/final_swapped_results/mistral_few.csv
Saved: /kaggle/working/final_swapped_results/mistral_zero.csv
Saved: /kaggle/working/final_swapped_results/phi_few.csv
Saved: /kaggle/working/final_swapped_results/phi_zero.csv
